# 🧠 Ternary Neural Networks — Phase 1 (CIFAR-10 / ResNet-18)
**Beyond Binary: Hardware-Aware AI**

### What's fixed in this version
The previous version collapsed to ~10% accuracy (random guessing) after quantization.
**Root cause:** BatchNorm statistics were calibrated for float weights.
After snapping weights to {−1, 0, +1}, activation magnitudes change completely —
BatchNorm's `running_mean` and `running_var` became stale and useless.

**Fix:** After ternarizing, run ~1000 images through the network in `train()` mode
so BatchNorm recomputes its statistics for the new ternary activation regime.
No weights are updated — just the normalization bookkeeping.

---
This notebook:
1. Trains a ResNet-18 baseline on CIFAR-10
2. Applies post-training ternary quantization {−1, 0, +1}
3. **Recalibrates BatchNorm** after each quantization
4. Evaluates accuracy across multiple thresholds
5. Measures sparsity and weight distribution
6. Visualizes results

## 1. Setup & Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
from torchvision.models import resnet18
from torch.utils.data import DataLoader, Subset
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import copy
import pandas as pd
from IPython.display import display

torch.manual_seed(42)
np.random.seed(42)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {DEVICE}')
if DEVICE.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Load CIFAR-10

In [ ]:
MEAN = (0.4914, 0.4822, 0.4465)
STD  = (0.2023, 0.1994, 0.2010)

train_transform = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True,  download=True, transform=train_transform)
test_dataset  = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=test_transform)

train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True,  num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=256, shuffle=False, num_workers=2, pin_memory=True)

# Small calibration loader — 2000 images is enough to re-estimate BN stats
calib_indices  = torch.randperm(len(train_dataset))[:2000].tolist()
calib_loader   = DataLoader(
    Subset(train_dataset, calib_indices),
    batch_size=128, shuffle=False, num_workers=2
)

CLASSES = ['airplane','automobile','bird','cat','deer',
           'dog','frog','horse','ship','truck']

print(f'Train samples       : {len(train_dataset):,}')
print(f'Test  samples       : {len(test_dataset):,}')
print(f'Calibration samples : {len(calib_indices):,}  ← for BN recalibration')

## 3. Build ResNet-18 (CIFAR-10 adapted)

In [ ]:
def build_resnet18_cifar():
    """
    ResNet-18 adapted for CIFAR-10 32×32 images:
      - First conv: 7×7 stride-2 → 3×3 stride-1  (preserves spatial resolution)
      - Remove initial MaxPool                     (avoids over-downsampling)
      - Final FC: 1000 → 10 classes
    """
    model = resnet18(weights=None)
    model.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc      = nn.Linear(model.fc.in_features, 10)
    return model


model = build_resnet18_cifar().to(DEVICE)

total_params  = sum(p.numel() for p in model.parameters())
weight_params = sum(p.numel() for n, p in model.named_parameters()
                    if 'weight' in n and 'bn' not in n)

print(f'Model              : ResNet-18 (CIFAR-10 adapted)')
print(f'Total params       : {total_params:,}')
print(f'Conv+Linear weights: {weight_params:,}  ← these get ternarized')
print(f'BatchNorm params   : {total_params - weight_params:,}  ← these are KEPT as float')

## 4. Train Baseline

In [ ]:
def train_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=(DEVICE.type == 'cuda')):
            out  = model(x)
            loss = criterion(out, y)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * y.size(0)
        correct    += out.argmax(1).eq(y).sum().item()
        total      += y.size(0)
    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        correct += model(x).argmax(1).eq(y).sum().item()
        total   += y.size(0)
    return correct / total


EPOCHS    = 15
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=5e-4)
scheduler = optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=0.1,
    epochs=EPOCHS, steps_per_epoch=len(train_loader)
)
scaler = torch.amp.GradScaler('cuda', enabled=(DEVICE.type == 'cuda'))

train_losses, train_accs, val_accs = [], [], []

print(f'{'Epoch':>6}  {'Train Loss':>11}  {'Train Acc':>10}  {'Val Acc':>9}  {'LR':>10}')
print('-' * 58)

for epoch in range(1, EPOCHS + 1):
    loss, tacc = train_epoch(model, train_loader, optimizer, criterion, scaler)
    vacc       = evaluate(model, test_loader)
    lr         = optimizer.param_groups[0]['lr']
    scheduler.step()

    train_losses.append(loss)
    train_accs.append(tacc)
    val_accs.append(vacc)

    print(f'{epoch:>6}  {loss:>11.4f}  {tacc*100:>9.2f}%  {vacc*100:>8.2f}%  {lr:>10.6f}')

baseline_accuracy = val_accs[-1]
print(f'\n✅ Baseline accuracy: {baseline_accuracy*100:.2f}%')

## 5. Training Curve

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.patch.set_facecolor('#0f0f0f')
BG, GRID   = '#1a1a1a', '#2a2a2a'
ACCENT, ACCENT2 = '#00ffcc', '#ff6b6b'

for ax in axes:
    ax.set_facecolor(BG)
    ax.spines[['top','right']].set_visible(False)
    ax.spines[['left','bottom']].set_color('#444')
    ax.tick_params(colors='#aaa')
    ax.grid(color=GRID, linestyle='--', linewidth=0.7, alpha=0.8)

epochs_x = range(1, EPOCHS + 1)
axes[0].plot(epochs_x, train_losses, color=ACCENT2, linewidth=2, label='Train Loss')
axes[0].set_title('Training Loss', color='white', fontweight='bold')
axes[0].set_xlabel('Epoch', color='#ccc')
axes[0].set_ylabel('Loss',  color='#ccc')
axes[0].legend(facecolor='#222', edgecolor='#444', labelcolor='white')

axes[1].plot(epochs_x, [a*100 for a in train_accs], color=ACCENT2, linewidth=2, label='Train Acc')
axes[1].plot(epochs_x, [a*100 for a in val_accs],   color=ACCENT,  linewidth=2, label='Val Acc')
axes[1].axhline(baseline_accuracy*100, color='white', linestyle=':', linewidth=1, alpha=0.5)
axes[1].set_title('Accuracy', color='white', fontweight='bold')
axes[1].set_xlabel('Epoch', color='#ccc')
axes[1].set_ylabel('Accuracy (%)', color='#ccc')
axes[1].legend(facecolor='#222', edgecolor='#444', labelcolor='white')

fig.suptitle('ResNet-18 on CIFAR-10 — Baseline Training', color='white', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('training_curve.png', dpi=150, bbox_inches='tight', facecolor=fig.get_facecolor())
plt.show()

## 6. Ternary Quantization + BatchNorm Recalibration

**Why recalibration is needed:**
- Float weights produce activations with a certain mean & variance
- BatchNorm's `running_mean` / `running_var` were fitted to those float activations
- Ternary weights produce completely different activation magnitudes
- Without recalibration → BatchNorm uses wrong stats → network outputs garbage (~10% accuracy)

**What recalibration does:**
- Puts BatchNorm into `train()` mode temporarily (starts collecting fresh stats)
- Runs 2000 images through the ternary network (no gradient, no weight update)
- BatchNorm updates `running_mean` / `running_var` to match ternary activations
- Switch back to `eval()` — now BatchNorm normalizes correctly for the ternary world

In [ ]:
def ternarize_weights(weight_tensor: torch.Tensor, alpha: float) -> torch.Tensor:
    """
    Threshold rule:
        t = alpha * max(|w|)
        w > +t  →  +1
        w < -t  →  -1
        else    →   0
    """
    t       = alpha * weight_tensor.abs().max()
    ternary = torch.zeros_like(weight_tensor)
    ternary[weight_tensor >  t] =  1.0
    ternary[weight_tensor < -t] = -1.0
    return ternary


def recalibrate_batchnorm(model: nn.Module, calib_loader: DataLoader, n_batches: int = 16):
    """
    Re-estimate BatchNorm running statistics for the ternary weight regime.

    Steps:
      1. Reset all BN running stats to zero
      2. Set model to train() mode  → BN starts collecting fresh stats
      3. Forward pass ~2000 images  → BN observes ternary activations
      4. Set model to eval() mode   → stats are now locked in

    No gradients, no weight updates — purely a statistics refresh.
    """
    # Step 1: Reset BN stats so old float stats don't bleed in
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.reset_running_stats()
            module.momentum = None   # use cumulative moving average for stability

    # Step 2 & 3: Collect fresh stats from ternary activations
    model.train()   # BN is now in stat-collection mode
    with torch.no_grad():
        for i, (x, _) in enumerate(calib_loader):
            if i >= n_batches:
                break
            model(x.to(DEVICE))    # forward only — no backward, no weight update

    # Step 4: Lock stats in — BN now normalizes correctly for ternary regime
    model.eval()


def apply_ternary_quantization(base_model: nn.Module, alpha: float,
                                calib_loader: DataLoader) -> nn.Module:
    """
    Full pipeline:
      1. Deep-copy baseline model
      2. Ternarize all Conv2d + Linear weights
      3. Recalibrate BatchNorm stats
    """
    t_model = copy.deepcopy(base_model)

    # Ternarize Conv2d and Linear weights only
    with torch.no_grad():
        for module in t_model.modules():
            if isinstance(module, (nn.Conv2d, nn.Linear)):
                module.weight.data = ternarize_weights(module.weight.data, alpha)

    # Recalibrate BatchNorm for the new activation regime
    recalibrate_batchnorm(t_model, calib_loader)

    return t_model


def compute_weight_stats(model: nn.Module) -> dict:
    """Per-layer and global ternary weight distribution."""
    layers, all_w = [], []
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            w     = module.weight.data.cpu().float().numpy().flatten()
            total = len(w)
            n_neg = int((w == -1).sum())
            n_zer = int((w ==  0).sum())
            n_pos = int((w ==  1).sum())
            layers.append({
                'layer':    name,
                'type':     type(module).__name__,
                'total':    total,
                'pct_neg':  100 * n_neg / total,
                'pct_zero': 100 * n_zer / total,
                'pct_pos':  100 * n_pos / total,
            })
            all_w.extend(w.tolist())
    all_w     = np.array(all_w)
    total_all = len(all_w)
    return {
        'layers': layers,
        'global': {
            'total':    total_all,
            'pct_neg':  100 * (all_w == -1).sum() / total_all,
            'pct_zero': 100 * (all_w ==  0).sum() / total_all,
            'pct_pos':  100 * (all_w ==  1).sum() / total_all,
        }
    }

print('✅ Quantization + BN recalibration functions ready.')

## 7. Threshold Sweep

In [ ]:
ALPHAS  = [0.05, 0.10, 0.15, 0.20, 0.25, 0.30]
results = []

print(f'Baseline accuracy: {baseline_accuracy*100:.2f}%\n')
print(f'{'Alpha':>6}  {'Accuracy':>9}  {'Δ Acc':>7}  {'% -1':>7}  {'% 0':>7}  {'% +1':>7}')
print('-' * 55)

for alpha in ALPHAS:
    # Quantize + recalibrate BN
    t_model = apply_ternary_quantization(model, alpha, calib_loader)
    acc     = evaluate(t_model, test_loader)
    stats   = compute_weight_stats(t_model)
    g       = stats['global']

    row = {
        'alpha':       alpha,
        'accuracy':    acc,
        'delta':       acc - baseline_accuracy,
        'pct_neg':     g['pct_neg'],
        'pct_zero':    g['pct_zero'],
        'pct_pos':     g['pct_pos'],
        'layer_stats': stats['layers'],
    }
    results.append(row)

    print(f'{alpha:>6.2f}  {acc*100:>8.2f}%  {(acc-baseline_accuracy)*100:>+7.2f}%  '
          f'{g["pct_neg"]:>7.1f}  {g["pct_zero"]:>7.1f}  {g["pct_pos"]:>7.1f}')

print('\n✅ Sweep complete.')

## 8. Results Table

In [ ]:
baseline_row = pd.DataFrame([{
    'Alpha': 'Baseline', 'Accuracy': f'{baseline_accuracy*100:.2f}%',
    'Δ Acc': '—', '% −1': '—', '% 0 (sparse)': '—', '% +1': '—',
}])
df = pd.DataFrame([{
    'Alpha':        r['alpha'],
    'Accuracy':     f"{r['accuracy']*100:.2f}%",
    'Δ Acc':        f"{r['delta']*100:+.2f}%",
    '% −1':         f"{r['pct_neg']:.1f}%",
    '% 0 (sparse)': f"{r['pct_zero']:.1f}%",
    '% +1':         f"{r['pct_pos']:.1f}%",
} for r in results])

df_full = pd.concat([baseline_row, df], ignore_index=True)
display(df_full.style
    .set_caption('Phase 1 — CIFAR-10 ResNet-18: Ternary Quantization Sweep (with BN Recalibration)')
    .set_properties(**{'text-align': 'center'}))

## 9. Visualizations

In [ ]:
alphas   = [r['alpha']        for r in results]
accs     = [r['accuracy']*100 for r in results]
sparsity = [r['pct_zero']     for r in results]
pct_neg  = [r['pct_neg']      for r in results]
pct_pos  = [r['pct_pos']      for r in results]

fig = plt.figure(figsize=(16, 12))
fig.patch.set_facecolor('#0f0f0f')
gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.42, wspace=0.35)

ACCENT, ACCENT2, ACCENT3 = '#00ffcc', '#ff6b6b', '#ffd166'
BG, GRID = '#1a1a1a', '#2a2a2a'

def style_ax(ax, title):
    ax.set_facecolor(BG)
    ax.spines[['top','right']].set_visible(False)
    ax.spines[['left','bottom']].set_color('#444')
    ax.tick_params(colors='#aaa')
    ax.yaxis.label.set_color('#ccc')
    ax.xaxis.label.set_color('#ccc')
    ax.set_title(title, color='white', fontsize=13, fontweight='bold', pad=10)
    ax.grid(color=GRID, linestyle='--', linewidth=0.7, alpha=0.8)

# ── Plot 1: Accuracy vs Alpha ──────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
ax1.axhline(baseline_accuracy*100, color=ACCENT2, linestyle='--',
            linewidth=1.5, label=f'Baseline ({baseline_accuracy*100:.2f}%)')
ax1.plot(alphas, accs, 'o-', color=ACCENT, linewidth=2, markersize=8,
         markerfacecolor=BG, markeredgewidth=2, label='Ternary (BN recalib.)')
for a, acc in zip(alphas, accs):
    ax1.annotate(f'{acc:.1f}%', (a, acc), textcoords='offset points',
                 xytext=(0, 10), ha='center', color='#eee', fontsize=9)
ax1.set_xlabel('Alpha (threshold coefficient)')
ax1.set_ylabel('Test Accuracy (%)')
ax1.legend(facecolor='#222', edgecolor='#444', labelcolor='white', fontsize=9)
style_ax(ax1, 'Accuracy vs Threshold (α)')

# ── Plot 2: Sparsity vs Alpha ──────────────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
ax2.fill_between(alphas, sparsity, alpha=0.15, color=ACCENT3)
ax2.plot(alphas, sparsity, 's-', color=ACCENT3, linewidth=2, markersize=8,
         markerfacecolor=BG, markeredgewidth=2, label='% Zero weights')
for a, s in zip(alphas, sparsity):
    ax2.annotate(f'{s:.1f}%', (a, s), textcoords='offset points',
                 xytext=(0, 10), ha='center', color='#eee', fontsize=9)
ax2.set_xlabel('Alpha (threshold coefficient)')
ax2.set_ylabel('Sparsity (% zero weights)')
ax2.legend(facecolor='#222', edgecolor='#444', labelcolor='white', fontsize=9)
style_ax(ax2, 'Sparsity vs Threshold (α)')

# ── Plot 3: Stacked weight distribution ───────────────────────────────────
ax3 = fig.add_subplot(gs[1, 0])
x = np.arange(len(alphas))
w = 0.55
ax3.bar(x, pct_neg,  width=w, label='−1 (negate)', color=ACCENT2,  alpha=0.9)
ax3.bar(x, sparsity, width=w, bottom=pct_neg, label='0 (skip)',    color=ACCENT3, alpha=0.9)
ax3.bar(x, pct_pos,  width=w,
        bottom=[n+z for n,z in zip(pct_neg, sparsity)],
        label='+1 (pass)', color=ACCENT, alpha=0.9)
ax3.set_xticks(x)
ax3.set_xticklabels([f'α={a}' for a in alphas], color='#aaa')
ax3.set_ylabel('Weight distribution (%)')
ax3.legend(facecolor='#222', edgecolor='#444', labelcolor='white', fontsize=9)
style_ax(ax3, 'Weight Distribution by Alpha')

# ── Plot 4: Accuracy–Sparsity trade-off ───────────────────────────────────
ax4 = fig.add_subplot(gs[1, 1])
sc = ax4.scatter(sparsity, accs, c=alphas, cmap='plasma',
                 s=120, zorder=5, edgecolors='white', linewidths=0.8)
for i, a in enumerate(alphas):
    ax4.annotate(f'α={a}', (sparsity[i], accs[i]),
                 textcoords='offset points', xytext=(8, 4),
                 color='#ccc', fontsize=9)
cbar = plt.colorbar(sc, ax=ax4)
cbar.set_label('Alpha', color='#ccc')
cbar.ax.yaxis.set_tick_params(color='#aaa')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='#aaa')
ax4.axhline(baseline_accuracy*100, color=ACCENT2, linestyle='--',
            linewidth=1.2, alpha=0.7, label='Baseline')
ax4.set_xlabel('Sparsity (% zero weights)')
ax4.set_ylabel('Test Accuracy (%)')
ax4.legend(facecolor='#222', edgecolor='#444', labelcolor='white', fontsize=9)
style_ax(ax4, 'Accuracy–Sparsity Trade-off')

fig.suptitle(
    'Ternary Neural Networks — Phase 1 Results (BN Recalibrated)\n'
    'CIFAR-10 ResNet-18: Post-Training Quantization {−1, 0, +1}',
    color='white', fontsize=15, fontweight='bold', y=1.01
)

plt.savefig('phase1_cifar10_results.png', dpi=150, bbox_inches='tight',
            facecolor=fig.get_facecolor())
plt.show()
print('✅ Saved: phase1_cifar10_results.png')

## 10. Per-Layer Breakdown (Best Alpha)

In [ ]:
candidates = [r for r in results if r['pct_zero'] > 50]
best = max(candidates, key=lambda r: r['accuracy']) if candidates else max(results, key=lambda r: r['accuracy'])

print(f"Best alpha (>50% sparse, highest accuracy): α = {best['alpha']}")
print(f"Accuracy : {best['accuracy']*100:.2f}%  (Δ {best['delta']*100:+.2f}%)")
print(f"Sparsity : {best['pct_zero']:.1f}% of conv/linear weights are zero\n")

layer_df = pd.DataFrame([{
    'Layer':      s['layer'],
    'Type':       s['type'],
    'Weights':    f"{s['total']:,}",
    '% −1':       f"{s['pct_neg']:.1f}%",
    '% 0 (skip)': f"{s['pct_zero']:.1f}%",
    '% +1':       f"{s['pct_pos']:.1f}%",
} for s in best['layer_stats']])

display(layer_df.style
    .set_caption(f"Layer-wise Stats — α = {best['alpha']}")
    .set_properties(**{'text-align': 'center'}))

## 11. Phase 1 Summary

In [ ]:
best_acc   = max(results, key=lambda r: r['accuracy'])
max_sparse = max(results, key=lambda r: r['pct_zero'])

print('=' * 64)
print('  PHASE 1 SUMMARY — CIFAR-10 / ResNet-18 (BN Recalibrated)')
print('=' * 64)
print(f'  Model              : ResNet-18 (CIFAR-10 adapted)')
print(f'  Dataset            : CIFAR-10 (60k images, 10 classes)')
print(f'  Training           : {EPOCHS} epochs, OneCycleLR, AMP')
print(f'  BN Recalibration   : 2,000 images post-quantization')
print()
print(f'  Baseline accuracy  : {baseline_accuracy*100:.2f}%')
print()
print(f'  Best ternary acc   : {best_acc["accuracy"]*100:.2f}%')
print(f'    → alpha = {best_acc["alpha"]}  |  Δ = {best_acc["delta"]*100:+.2f}%')
print()
print(f'  Max sparsity       : {max_sparse["pct_zero"]:.1f}% zero weights')
print(f'    → alpha = {max_sparse["alpha"]}  |  acc = {max_sparse["accuracy"]*100:.2f}%')
print()
print(f'  Hardware implications (Phase 2+):')
print(f'    • {max_sparse["pct_zero"]:.0f}% of conv multiplications → SKIP   (w = 0)')
print(f'    • {max_sparse["pct_pos"]:.0f}% of conv multiplications → PASS   (w = +1)')
print(f'    • {max_sparse["pct_neg"]:.0f}% of conv multiplications → NEGATE (w = −1)')
print()
print('  ✅ Phase 1 COMPLETE — feasibility on realistic CNN confirmed.')
print('=' * 64)